#Part 2:

##1. For each year & month

In [0]:
%sql
USE bde;

WITH base AS (
  SELECT
    date_format(pickup_datetime,'yyyy-MM')              AS ym,
    dayofweek(pickup_datetime)                          AS dow_num,
    date_format(pickup_datetime,'EEEE')                 AS dow_name,
    hour(pickup_datetime)                               AS hr,
    passenger_count,
    total_amount
  FROM bde.trips_final
),

m_agg AS (
  SELECT
    ym,
    COUNT(*)                                  AS total_trips,
    ROUND(AVG(passenger_count), 2)            AS avg_passengers,
    ROUND(AVG(total_amount), 2)               AS avg_total_paid_per_trip,
    ROUND(AVG(total_amount / NULLIF(passenger_count,0)), 2) AS avg_paid_per_passenger
  FROM base
  GROUP BY ym
),

m_dow AS (
  SELECT ym, dow_name AS busiest_dow
  FROM (
    SELECT
      ym, dow_name, dow_num,
      ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, dow_num) AS rn
    FROM base
    GROUP BY ym, dow_name, dow_num
  ) t
  WHERE rn = 1
),

m_hr AS (
  SELECT ym, hr AS busiest_hour
  FROM (
    SELECT
      ym, hr,
      ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, hr) AS rn
    FROM base
    GROUP BY ym, hr
  ) t
  WHERE rn = 1
)

SELECT
  a.ym,
  a.total_trips,
  d.busiest_dow,                        
  h.busiest_hour,                        
  a.avg_passengers,                      
  a.avg_total_paid_per_trip,            
  a.avg_paid_per_passenger              
FROM m_agg a
JOIN m_dow d USING (ym)
JOIN m_hr  h USING (ym)
ORDER BY a.ym;

ym,total_trips,busiest_dow,busiest_hour,avg_passengers,avg_total_paid_per_trip,avg_paid_per_passenger
2009-01,1222,Thursday,0,1.6,17.82,15.03
2010-09,203,Thursday,1,1.05,17.41,16.88
2011-01,2,Monday,23,1.0,12.8,12.8
2011-02,1,Tuesday,0,1.0,15.8,15.8
2012-09,3,Thursday,19,1.0,9.79,9.79
2014-01,14438389,Friday,19,1.69,14.34,11.63
2014-02,13925644,Saturday,19,1.68,14.48,11.78
2014-03,16553179,Saturday,19,1.68,14.64,11.91
2014-04,15770482,Wednesday,19,1.68,14.93,12.12
2014-05,16026941,Friday,19,1.68,15.5,12.55


Insight: A monthly summary (year–month) with total trips, busiest day of week, busiest hour, average passengers, average paid per trip, and average paid per passenger states that the demand is highly seasonal with weekend and evening peaks; scale driver supply and price sensitivity around those windows to capture the bulk of revenue with minimal idle time.

##2. For each taxi colour (yellow and green), what was the average, median, minimum and maximum trip 1. duration in minutes ; 2. distance in km ; 3. speed in km per hour:

In [0]:
%sql
USE bde;

WITH feats AS (
  SELECT
    color,
    (unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60.0 AS dur_min,
    trip_distance * 1.60934 AS dist_km,
    CASE
      WHEN unix_timestamp(dropoff_datetime) > unix_timestamp(pickup_datetime)
      THEN (trip_distance * 1.60934) /
           ((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 3600.0)
    END AS speed_kmh
  FROM bde.trips_final
)
SELECT
  color,

  ROUND(AVG(dur_min), 2)                                  AS duration_avg_min,
  ROUND(percentile_approx(dur_min, 0.5, 10000), 2)        AS duration_median_min,
  ROUND(MIN(dur_min), 2)                                  AS duration_min_min,
  ROUND(MAX(dur_min), 2)                                  AS duration_max_min,

  ROUND(AVG(dist_km), 2)                                  AS distance_avg_km,
  ROUND(percentile_approx(dist_km, 0.5, 10000), 2)        AS distance_median_km,
  ROUND(MIN(dist_km), 2)                                  AS distance_min_km,
  ROUND(MAX(dist_km), 2)                                  AS distance_max_km,

  ROUND(AVG(speed_kmh), 2)                                AS speed_avg_kmh,
  ROUND(percentile_approx(speed_kmh, 0.5, 10000), 2)      AS speed_median_kmh,
  ROUND(MIN(speed_kmh), 2)                                AS speed_min_kmh,
  ROUND(MAX(speed_kmh), 2)                                AS speed_max_kmh

FROM feats
GROUP BY color
ORDER BY color;

color,duration_avg_min,duration_median_min,duration_min_min,duration_max_min,distance_avg_km,distance_median_km,distance_min_km,distance_max_km,speed_avg_kmh,speed_median_kmh,speed_min_kmh,speed_max_kmh
green,13.52,10.50,1.00,180.00,4.75,3.09,0.11,49.99,20.4,18.54,0.04,99.99
yellow,14.36,11.25,1.00,180.00,4.87,2.74,0.11,49.99,18.88,16.58,0.04,100.0


Insight: for every Taxi colour, Trip duration (minutes), distance (km), speed (km/h) features and summary distributions/percentiles display that most rides are short and slow (urban traffic), so efficiency gains come from reducing deadhead time and improving pickup positioning rather than chasing longer trips.

##3. For each taxi colour (yellow and green), each pair of pickup and drop off locations (use boroughs), each year-month, each day of week and each hour:

In [0]:
%sql
USE bde;

SELECT
  color,
  pu_borough,
  do_borough,
  month(pickup_datetime)              AS month_num,
  date_format(pickup_datetime,'EEEE') AS day_of_week,
  hour(pickup_datetime)               AS hour_of_day,

  COUNT(*)                            AS total_trips,             
  ROUND(AVG(trip_distance * 1.60934), 2) AS avg_distance_km,      
  ROUND(AVG(total_amount), 2)         AS avg_total_amount, 
  ROUND(SUM(total_amount), 2)         AS total_amnt           

FROM bde.trips_final
GROUP BY
  color,
  pu_borough,
  do_borough,
  month(pickup_datetime),
  date_format(pickup_datetime,'EEEE'),
  hour(pickup_datetime)
ORDER BY
  color, pu_borough, do_borough, month_num, day_of_week, hour_of_day;

color,pu_borough,do_borough,month_num,day_of_week,hour_of_day,total_trips,avg_distance_km,avg_total_amount,total_amnt
green,Bronx,Bronx,1,Friday,0,1126,3.64,10.44,11759.92
green,Bronx,Bronx,1,Friday,1,987,3.64,10.7,10557.0
green,Bronx,Bronx,1,Friday,2,709,3.9,11.23,7959.46
green,Bronx,Bronx,1,Friday,3,527,4.19,11.21,5907.38
green,Bronx,Bronx,1,Friday,4,453,4.33,11.58,5244.91
green,Bronx,Bronx,1,Friday,5,457,4.41,10.88,4970.66
green,Bronx,Bronx,1,Friday,6,750,4.54,11.26,8446.33
green,Bronx,Bronx,1,Friday,7,2242,3.97,11.27,25261.45
green,Bronx,Bronx,1,Friday,8,3526,3.73,11.28,39775.91
green,Bronx,Bronx,1,Friday,9,2752,3.95,11.18,30760.48


Insight: The reference table by (color, PU borough, DO borough, month, day, hour) with total trips, average distance, avg amount paid per trip and total amount gives a stable “expected fare” prior for each spatiotemporal slice; it’s ideal for benchmarking models and detecting shifts in rider mix.

####Saving Q3c result for further processing (code below):

In [0]:
%sql
USE bde;

SELECT
  color,
  pu_borough,
  do_borough,
  month(pickup_datetime)              AS month_num,
  date_format(pickup_datetime,'EEEE') AS day_of_week,
  hour(pickup_datetime)               AS hour_of_day,
  ROUND(AVG(total_amount), 2)         AS avg_total_amount
FROM bde.trips_final
WHERE pu_borough IS NOT NULL
  AND do_borough IS NOT NULL  
GROUP BY
  color,
  pu_borough,
  do_borough,
  month(pickup_datetime),
  date_format(pickup_datetime,'EEEE'),
  hour(pickup_datetime)
ORDER BY
  color, pu_borough, do_borough, month_num, day_of_week, hour_of_day;

color,pu_borough,do_borough,month_num,day_of_week,hour_of_day,avg_total_amount
green,Bronx,Bronx,1,Friday,0,10.44
green,Bronx,Bronx,1,Friday,1,10.7
green,Bronx,Bronx,1,Friday,2,11.23
green,Bronx,Bronx,1,Friday,3,11.21
green,Bronx,Bronx,1,Friday,4,11.58
green,Bronx,Bronx,1,Friday,5,10.88
green,Bronx,Bronx,1,Friday,6,11.26
green,Bronx,Bronx,1,Friday,7,11.27
green,Bronx,Bronx,1,Friday,8,11.28
green,Bronx,Bronx,1,Friday,9,11.18


Note: This result was downloaded (as CSV) manually and uploaded at Path: "/Volumes/workspace/bde/assignment2/" and named as: "Part_2_3_c.csv" for further process (in Part 3)

##4. For 2024, compute the share of total revenue contributed by the top 10 "pickup & dropoff borough" pairs (ranked by total_amount)

In [0]:
%sql

WITH y24 AS (
  SELECT pu_borough, do_borough, total_amount
  FROM bde.trips_final
  WHERE year(pickup_datetime) = 2024
),
pair_rev AS (
  SELECT
    pu_borough,
    do_borough,
    SUM(total_amount) AS revenue
  FROM y24
  GROUP BY pu_borough, do_borough
),
tot AS (
  SELECT SUM(revenue) AS total_rev FROM pair_rev
),
ranked AS (
  SELECT
    p.*,
    RANK() OVER (ORDER BY revenue DESC) AS rk
  FROM pair_rev p
)
SELECT
  r.pu_borough,
  r.do_borough,
  ROUND(r.revenue, 2) AS revenue_usd,
  ROUND(r.revenue / t.total_rev * 100, 2) AS share_prcnt,
  r.rk
FROM ranked r
CROSS JOIN tot t
WHERE rk <= 10
ORDER BY rk;

pu_borough,do_borough,revenue_usd,share_prcnt,rk
Manhattan,Manhattan,6.2966733258E8,61.74,1
Queens,Manhattan,1.6858183168E8,16.53,2
Manhattan,Queens,6.792557353E7,6.66,3
Queens,Brooklyn,3.674751489E7,3.6,4
Manhattan,Brooklyn,3.124339379E7,3.06,5
Queens,Queens,2.968857529E7,2.91,6
Manhattan,EWR,1.16811543E7,1.15,7
Queens,Unknown,9261353.73,0.91,8
Manhattan,Unknown,5895442.04,0.58,9
Queens,Bronx,5384850.96,0.53,10


Insight: A small set of borough flows dominates revenue, prioritize supply balancing and incentive tuning on these corridors first for outsized impact.

##5. What was the percentage of trips where drivers received tips?

In [0]:
%sql

SELECT
  ROUND((100.0 * SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END)) / COUNT(*), 2) 
    AS prcnt_trips_with_tips
FROM bde.trips_final;

prcnt_trips_with_tips
63.05


Insight: Overall share of trips that include any tip (63.05%).

##6. For tips received, what was the percentage where the driver received tips of at least $15

In [0]:
%sql

SELECT
  ROUND(
    100.0 * SUM(CASE WHEN tip_amount >= 15 THEN 1 ELSE 0 END) 
    / SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END), 2
  ) AS prcnt_tips_grtr_thn_15
FROM bde.trips_final;

prcnt_tips_grtr_thn_15
0.80


Insight: Among tipped trips, the percentage where the driver received tips of at least $15 was 0.80%

##7. Classify each trip into bins of durations and calculate:
Average speed (km per hour) & 
Average distance per dollar (km per $)


In [0]:
%sql

WITH base AS (
  SELECT
    (unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60.0 AS dur_min,
    trip_distance * 1.60934 AS dist_km,
    total_amount
  FROM bde.trips_final
),
binned AS (
  SELECT
    CASE
      WHEN dur_min < 5   THEN 'Under 5 mins'
      WHEN dur_min < 10  THEN '5–10 mins'
      WHEN dur_min < 20  THEN '10–20 mins'
      WHEN dur_min < 30  THEN '20–30 mins'
      WHEN dur_min < 60  THEN '30–60 mins'
      ELSE                   'At least 60 mins'
    END AS duration_bin,
    CASE WHEN dur_min > 0 THEN dist_km / (dur_min / 60.0) END AS speed_kmh,
    CASE WHEN total_amount > 0 THEN dist_km / total_amount END AS km_per_dollar
  FROM base
)
SELECT
  duration_bin,
  ROUND(AVG(speed_kmh), 2)      AS avg_speed_kmh,
  ROUND(AVG(km_per_dollar), 4)  AS avg_km_per_dollar
FROM binned
GROUP BY duration_bin
ORDER BY CASE duration_bin
           WHEN 'Under 5 mins'   THEN 1
           WHEN '5–10 mins'      THEN 2
           WHEN '10–20 mins'     THEN 3
           WHEN '20–30 mins'     THEN 4
           WHEN '30–60 mins'     THEN 5
           WHEN 'At least 60 mins' THEN 6
         END;

duration_bin,avg_speed_kmh,avg_km_per_dollar
Under 5 mins,19.75,0.1629
5–10 mins,17.2,0.2105
10–20 mins,17.9,0.2574
20–30 mins,21.35,0.305
30–60 mins,25.73,0.3699
At least 60 mins,22.08,0.5035


Insight: Short-to-medium bins usually maximize revenue/hour and keep tip rates healthy—optimize positioning to keep cars in these bins rather than chasing rare, long hauls.

##8. Which duration bin should taxi driver target to maximum his income?

In [0]:
%sql
USE bde;

WITH base AS (
  SELECT
    (unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60.0 AS dur_min,
    trip_distance * 1.60934 AS dist_km,
    total_amount,
    tip_amount
  FROM bde.trips_final
),
binned AS (
  SELECT
    CASE
      WHEN dur_min < 5   THEN 'Under 5 mins'
      WHEN dur_min < 10  THEN '5–10 mins'
      WHEN dur_min < 20  THEN '10–20 mins'
      WHEN dur_min < 30  THEN '20–30 mins'
      WHEN dur_min < 60  THEN '30–60 mins'
      ELSE                   'At least 60 mins'
    END AS duration_bin,
    dur_min,
    dist_km,
    total_amount,
    tip_amount,
    
    CASE WHEN dur_min > 0 THEN dist_km / (dur_min / 60.0) END AS speed_kmh,
    
    CASE WHEN total_amount > 0 THEN dist_km / total_amount END AS km_per_dollar
  FROM base
)
SELECT
  duration_bin,

  ROUND(AVG(total_amount), 2)                                     AS avg_total_per_trip_usd,
  ROUND(AVG(dur_min), 2)                                          AS avg_duration_min,
  ROUND(SUM(total_amount) / NULLIF(SUM(dur_min) / 60.0, 0), 2)    AS revenue_per_hour_usd,  

  ROUND(100.0 * SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) / COUNT(*), 2)
      AS tip_rate_pct,
  ROUND(AVG(tip_amount), 2)                                       AS avg_tip_usd,

  ROUND(AVG(speed_kmh), 2)                                        AS avg_speed_kmh,
  ROUND(AVG(km_per_dollar), 4)                                    AS avg_km_per_dollar,

  COUNT(*)                                                        AS trips_in_bin,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)              AS trip_share_pct

FROM binned
GROUP BY duration_bin
ORDER BY revenue_per_hour_usd DESC;

duration_bin,avg_total_per_trip_usd,avg_duration_min,revenue_per_hour_usd,tip_rate_pct,avg_tip_usd,avg_speed_kmh,avg_km_per_dollar,trips_in_bin,trip_share_pct
Under 5 mins,7.25,3.53,123.2,54.91,0.77,19.75,0.1629,134172829,13.98
5–10 mins,10.21,7.42,82.59,61.85,1.12,17.2,0.2105,285003099,29.69
30–60 mins,47.28,39.27,72.23,66.07,5.28,25.73,0.3699,70843613,7.38
10–20 mins,16.21,14.18,68.6,65.49,1.79,17.9,0.2574,339611747,35.38
20–30 mins,27.48,24.11,68.41,66.53,3.11,21.35,0.305,122170103,12.73
At least 60 mins,68.42,73.23,56.06,58.68,7.05,22.08,0.5035,7990578,0.83


Analysis: The taxi driver should primarily target trips in the 5–10 minute duration bin, as they offer the best balance of high revenue per hour and large trip share (with steady demand). Very short trips (<5 mins) give slightly higher hourly returns but occur less often, while longer trips reduce income efficiency.